# Jewellery Site Chatbot — Backend Test Notebook

This notebook is self-contained: it writes out all backend files, installs
dependencies, starts the FastAPI server in-process, and runs through every
endpoint so you can validate the full pipeline before moving to VS Code
for the real deployment.

**Run cells top to bottom.** Set your `GROQ_API_KEY` in the cell marked below
before running anything that calls Groq.

## 1. Install dependencies

In [ ]:
!pip install fastapi uvicorn python-multipart aiohttp beautifulsoup4 lxml playwright groq nest-asyncio -q
!playwright install --with-deps chromium

## 2. Write backend files
Each cell below writes one module to disk in this Colab session.

In [ ]:
%%writefile scraper.py
"""
scraper.py
==========
Site ingestion module for the per-site jewellery chatbot.

Given a store URL, produces a normalized dict:
{
    "site_url": str,
    "products": [
        {
            "id": str,
            "title": str,
            "price": float | None,
            "currency": str | None,
            "image_url": str | None,
            "product_url": str,
            "description": str,
        },
        ...
    ],
    "policies": {
        "return_policy": str, "shipping_policy": str, "faq": str,
        "terms": str, "other": str,
    },
    "meta": {
        "method": "shopify_json" | "html_fallback" | "browser_fallback",
        "product_count": int,
        "elapsed_seconds": float,
    }
}

Three-tier strategy
--------------------
1. Shopify JSON fast-path — /products.json, works for any Shopify storefront
   regardless of theme. Fast, structured, no HTML parsing needed.
2. Static HTML crawl — for server-rendered non-Shopify sites. Fetches the
   homepage, discovers both direct product links AND collection/category
   links, then crawls one level into those collection pages too (so it
   doesn't stop at whatever happens to be linked from the homepage).
3. Headless-browser fallback (Playwright) — only triggered when tiers 1+2
   together find fewer than MIN_PRODUCTS_FOR_SUCCESS products. This catches
   JavaScript-rendered SPA sites (React/Angular/Vue storefronts) where the
   raw HTML has no real content until JS executes. Runs a single headless
   Chromium instance, bounded concurrency via a semaphore over browser
   pages/contexts. This is safe on Hugging Face Spaces free tier (2 vCPU /
   16GB RAM) — the RAM problems in earlier iterations of this project came
   from combining Playwright with CLIP/torch/FAISS on a 512MB Render
   instance; neither of those is used anymore.

Designed to run:
  - Standalone in a Colab cell (call `scrape_site_sync(url)`)
  - Inside a FastAPI async endpoint (call `await scrape_site(url)`)

Colab setup:
    !pip install aiohttp beautifulsoup4 lxml playwright -q
    !playwright install --with-deps chromium
"""

import asyncio
import json
import re
import time
from urllib.parse import urljoin, urlparse

import aiohttp
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------

DEFAULT_TIMEOUT_SECONDS = 35
MAX_PRODUCTS = 200
MAX_CONCURRENT_REQUESTS = 8
MAX_CONCURRENT_BROWSER_PAGES = 4
MIN_PRODUCTS_FOR_SUCCESS = 5          # below this, tier 3 (browser) kicks in
MAX_COLLECTION_PAGES_TO_CRAWL = 6     # tier 2 depth-2 crawl cap
REQUEST_TIMEOUT = aiohttp.ClientTimeout(total=10)
USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
HEADERS = {"User-Agent": USER_AGENT}

POLICY_KEYWORDS = {
    "return_policy": ["return", "refund", "exchange"],
    "shipping_policy": ["shipping", "delivery"],
    "faq": ["faq", "help", "support"],
    "terms": ["terms", "condition", "privacy"],
}

PRODUCT_LINK_HINTS = ["/product", "/products", "/item", "/p/"]
COLLECTION_LINK_HINTS = ["/collections", "/collection", "/category", "/categories", "/shop", "/catalog"]

PRICE_REGEX = re.compile(r"[\d,]+\.?\d*")


# ---------------------------------------------------------------------------
# Public entry points
# ---------------------------------------------------------------------------

def scrape_site_sync(url: str, timeout: int = DEFAULT_TIMEOUT_SECONDS) -> dict:
    """Synchronous wrapper for Colab cells / quick testing."""
    return asyncio.run(scrape_site(url, timeout=timeout))


async def scrape_site(url: str, timeout: int = DEFAULT_TIMEOUT_SECONDS) -> dict:
    """Main entry point. Tries Shopify -> static HTML crawl -> headless browser."""
    start = time.time()
    url = _normalize_base_url(url)

    async with aiohttp.ClientSession(headers=HEADERS, timeout=REQUEST_TIMEOUT) as session:
        products: list = []
        policies = _empty_policies()
        method = "shopify_json"

        # ---- Tier 1: Shopify fast-path ----
        try:
            shopify_result = await asyncio.wait_for(
                _try_shopify(url, session), timeout=max(3, timeout * 0.25)
            )
        except (asyncio.TimeoutError, Exception):
            shopify_result = None

        if shopify_result and shopify_result["products"]:
            products = shopify_result["products"]
            try:
                policies = await asyncio.wait_for(
                    _scrape_policies(url, session), timeout=max(3, timeout * 0.2)
                )
            except (asyncio.TimeoutError, Exception):
                pass

        # ---- Tier 2: static HTML crawl (if Shopify found nothing) ----
        if not products:
            method = "html_fallback"
            elapsed = time.time() - start
            remaining = max(5, timeout * 0.55 - elapsed)
            try:
                html_result = await asyncio.wait_for(
                    _html_fallback_scrape(url, session), timeout=remaining
                )
                products = html_result["products"]
                policies = html_result["policies"]
            except (asyncio.TimeoutError, Exception):
                pass

        # ---- Tier 3: headless browser (if still too few products) ----
        if len(products) < MIN_PRODUCTS_FOR_SUCCESS:
            elapsed = time.time() - start
            remaining = max(8, timeout - elapsed)
            try:
                browser_result = await asyncio.wait_for(
                    _browser_fallback_scrape(url), timeout=remaining
                )
                if len(browser_result["products"]) > len(products):
                    products = browser_result["products"]
                    method = "browser_fallback"
                if not any(policies.values()) and any(browser_result["policies"].values()):
                    policies = browser_result["policies"]
            except (asyncio.TimeoutError, Exception):
                pass

        return {
            "site_url": url,
            "products": products[:MAX_PRODUCTS],
            "policies": policies,
            "meta": {
                "method": method,
                "product_count": len(products[:MAX_PRODUCTS]),
                "elapsed_seconds": round(time.time() - start, 1),
            },
        }


# ---------------------------------------------------------------------------
# Tier 1: Shopify fast-path
# ---------------------------------------------------------------------------

async def _try_shopify(base_url: str, session: aiohttp.ClientSession) -> dict | None:
    products = []
    page = 1
    while len(products) < MAX_PRODUCTS:
        endpoint = f"{base_url}/products.json?limit=250&page={page}"
        try:
            async with session.get(endpoint) as resp:
                if resp.status != 200:
                    break
                data = await resp.json(content_type=None)
        except Exception:
            break

        batch = data.get("products", [])
        if not batch:
            break

        for p in batch:
            variant = (p.get("variants") or [{}])[0]
            image = (p.get("images") or [{}])[0].get("src") if p.get("images") else None
            products.append({
                "id": str(p.get("id")),
                "title": p.get("title", ""),
                "price": _safe_float(variant.get("price")),
                "currency": None,
                "image_url": image,
                "product_url": f"{base_url}/products/{p.get('handle')}",
                "description": _strip_html(p.get("body_html", ""))[:500],
            })

        if len(batch) < 250:
            break
        page += 1

    if not products:
        return None
    return {"site_url": base_url, "products": products[:MAX_PRODUCTS]}


# ---------------------------------------------------------------------------
# Tier 2: static HTML crawl (homepage + one level into collection pages)
# ---------------------------------------------------------------------------

async def _html_fallback_scrape(base_url: str, session: aiohttp.ClientSession) -> dict:
    homepage_html = await _fetch_text(session, base_url)
    if homepage_html is None:
        return {"site_url": base_url, "products": [], "policies": _empty_policies()}

    soup = BeautifulSoup(homepage_html, "html.parser")
    all_links = _extract_links(soup, base_url)

    direct_product_links = {l for l in all_links if any(h in l.lower() for h in PRODUCT_LINK_HINTS)}
    collection_links = list({l for l in all_links if any(h in l.lower() for h in COLLECTION_LINK_HINTS)})[:MAX_COLLECTION_PAGES_TO_CRAWL]

    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    # crawl one level into collection pages to discover more product links
    collection_htmls = await asyncio.gather(
        *[_fetch_text_bounded(session, link, sem) for link in collection_links],
        return_exceptions=True,
    )
    for html in collection_htmls:
        if isinstance(html, str):
            csoup = BeautifulSoup(html, "html.parser")
            clinks = _extract_links(csoup, base_url)
            direct_product_links |= {l for l in clinks if any(h in l.lower() for h in PRODUCT_LINK_HINTS)}

    product_links = list(direct_product_links)[:MAX_PRODUCTS]
    policy_links = _classify_policy_links(all_links)

    products_results = await asyncio.gather(
        *[_fetch_and_parse_product(session, link, sem) for link in product_links],
        return_exceptions=True,
    )
    policies = await _scrape_policies_from_links(session, policy_links, sem)
    products = [p for p in products_results if isinstance(p, dict) and p]

    return {"site_url": base_url, "products": products, "policies": policies}


async def _fetch_and_parse_product(session, url, sem) -> dict | None:
    html = await _fetch_text_bounded(session, url, sem)
    if html is None:
        return None
    return _parse_product_html(html, url)


def _parse_product_html(html: str, url: str) -> dict | None:
    soup = BeautifulSoup(html, "html.parser")

    ld_product = _parse_json_ld_product(soup, url)
    if ld_product:
        return ld_product

    title = _meta_content(soup, "og:title") or (soup.title.string if soup.title else "")
    image = _meta_content(soup, "og:image")
    price = _heuristic_price(soup)
    description = _meta_content(soup, "og:description") or ""

    if not title:
        return None

    return {
        "id": url,
        "title": title.strip(),
        "price": price,
        "currency": None,
        "image_url": image,
        "product_url": url,
        "description": description[:500],
    }


def _parse_json_ld_product(soup: BeautifulSoup, url: str) -> dict | None:
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "")
        except Exception:
            continue
        candidates = data if isinstance(data, list) else [data]
        for entry in candidates:
            if not isinstance(entry, dict):
                continue
            if entry.get("@type") not in ("Product", ["Product"]):
                continue
            offers = entry.get("offers", {})
            if isinstance(offers, list):
                offers = offers[0] if offers else {}
            price = _safe_float(offers.get("price")) if isinstance(offers, dict) else None
            currency = offers.get("priceCurrency") if isinstance(offers, dict) else None
            image = entry.get("image")
            if isinstance(image, list):
                image = image[0] if image else None
            return {
                "id": entry.get("sku") or url,
                "title": entry.get("name", ""),
                "price": price,
                "currency": currency,
                "image_url": image,
                "product_url": url,
                "description": (entry.get("description") or "")[:500],
            }
    return None


# ---------------------------------------------------------------------------
# Tier 3: headless-browser fallback (Playwright) for JS-rendered sites
# ---------------------------------------------------------------------------

async def _browser_fallback_scrape(base_url: str) -> dict:
    from playwright.async_api import async_playwright

    products: list = []
    policies = _empty_policies()

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--disable-gpu", "--no-sandbox"])
        try:
            context = await browser.new_context(user_agent=USER_AGENT)
            home_page = await context.new_page()
            await home_page.goto(base_url, wait_until="networkidle", timeout=15000)
            homepage_html = await home_page.content()
            await home_page.close()

            soup = BeautifulSoup(homepage_html, "html.parser")
            all_links = _extract_links(soup, base_url)
            direct_product_links = {l for l in all_links if any(h in l.lower() for h in PRODUCT_LINK_HINTS)}
            collection_links = list({l for l in all_links if any(h in l.lower() for h in COLLECTION_LINK_HINTS)})[:MAX_COLLECTION_PAGES_TO_CRAWL]
            policy_links = _classify_policy_links(all_links)

            sem = asyncio.Semaphore(MAX_CONCURRENT_BROWSER_PAGES)

            # render collection pages to discover more product links
            collection_htmls = await asyncio.gather(
                *[_render_page(context, link, sem) for link in collection_links],
                return_exceptions=True,
            )
            for html in collection_htmls:
                if isinstance(html, str):
                    csoup = BeautifulSoup(html, "html.parser")
                    clinks = _extract_links(csoup, base_url)
                    direct_product_links |= {l for l in clinks if any(h in l.lower() for h in PRODUCT_LINK_HINTS)}

            product_links = list(direct_product_links)[:MAX_PRODUCTS]

            product_htmls = await asyncio.gather(
                *[_render_page(context, link, sem) for link in product_links],
                return_exceptions=True,
            )
            for html, link in zip(product_htmls, product_links):
                if isinstance(html, str):
                    parsed = _parse_product_html(html, link)
                    if parsed:
                        products.append(parsed)

            # render policy pages (bounded)
            for category, link in policy_links.items():
                html = await _render_page(context, link, sem)
                if html:
                    psoup = BeautifulSoup(html, "html.parser")
                    policies[category] = psoup.get_text(separator=" ", strip=True)[:3000]

        finally:
            await browser.close()

    return {"site_url": base_url, "products": products, "policies": policies}


async def _render_page(context, url: str, sem: asyncio.Semaphore) -> str | None:
    async with sem:
        page = await context.new_page()
        try:
            await page.goto(url, wait_until="networkidle", timeout=12000)
            return await page.content()
        except Exception:
            return None
        finally:
            await page.close()


# ---------------------------------------------------------------------------
# Policy scraping (static tier)
# ---------------------------------------------------------------------------

async def _scrape_policies(base_url: str, session: aiohttp.ClientSession) -> dict:
    homepage_html = await _fetch_text(session, base_url)
    if homepage_html is None:
        return _empty_policies()
    soup = BeautifulSoup(homepage_html, "html.parser")
    all_links = _extract_links(soup, base_url)
    policy_links = _classify_policy_links(all_links)
    sem = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)
    return await _scrape_policies_from_links(session, policy_links, sem)


async def _scrape_policies_from_links(session, policy_links: dict, sem) -> dict:
    policies = _empty_policies()
    for category, url in policy_links.items():
        html = await _fetch_text_bounded(session, url, sem)
        if html:
            soup = BeautifulSoup(html, "html.parser")
            policies[category] = soup.get_text(separator=" ", strip=True)[:3000]
    return policies


def _classify_policy_links(all_links: set) -> dict:
    result = {}
    for link in all_links:
        lower = link.lower()
        for category, keywords in POLICY_KEYWORDS.items():
            if category in result:
                continue
            if any(kw in lower for kw in keywords):
                result[category] = link
    return result


def _empty_policies() -> dict:
    return {"return_policy": "", "shipping_policy": "", "faq": "", "terms": "", "other": ""}


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

async def _fetch_text(session: aiohttp.ClientSession, url: str) -> str | None:
    try:
        async with session.get(url) as resp:
            if resp.status != 200:
                return None
            return await resp.text()
    except Exception:
        return None


async def _fetch_text_bounded(session: aiohttp.ClientSession, url: str, sem: asyncio.Semaphore) -> str | None:
    async with sem:
        return await _fetch_text(session, url)


def _extract_links(soup: BeautifulSoup, base_url: str) -> set:
    links = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(base_url, href)
        if urlparse(full).netloc == urlparse(base_url).netloc:
            links.add(full.split("#")[0])
    return links


def _meta_content(soup: BeautifulSoup, prop: str) -> str | None:
    tag = soup.find("meta", property=prop) or soup.find("meta", attrs={"name": prop})
    return tag["content"] if tag and tag.has_attr("content") else None


def _heuristic_price(soup: BeautifulSoup) -> float | None:
    for selector in [".price", "[class*=price]", "[itemprop=price]"]:
        el = soup.select_one(selector)
        if el:
            match = PRICE_REGEX.search(el.get_text())
            if match:
                return _safe_float(match.group().replace(",", ""))
    return None


def _safe_float(value) -> float | None:
    try:
        return float(str(value).replace(",", ""))
    except (TypeError, ValueError):
        return None


def _strip_html(html: str) -> str:
    return BeautifulSoup(html or "", "html.parser").get_text(separator=" ", strip=True)


def _normalize_base_url(url: str) -> str:
    url = url.strip()
    if not url.startswith(("http://", "https://")):
        url = "https://" + url
    parsed = urlparse(url)
    return f"{parsed.scheme}://{parsed.netloc}"


# ---------------------------------------------------------------------------
# Quick manual test (Colab)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    test_url = "https://www.tanishq.co.in"
    data = scrape_site_sync(test_url)
    print(f"Method: {data['meta']['method']}, Products found: {data['meta']['product_count']}")
    print(json.dumps(data["products"][:2], indent=2))


In [ ]:
%%writefile session_store.py
"""
session_store.py
=================
In-memory, multi-user session storage. No database — each session holds
one user's scraped site (catalog + policies) plus their chat history.

Sessions are keyed by a UUID handed to the frontend after site ingestion.
A background cleanup task drops sessions inactive past SESSION_TTL_SECONDS
so memory doesn't grow unbounded across many concurrent users.

Designed to be a single module-level store shared by the FastAPI app
(import `store` and use it directly — no class instantiation needed
per-request).
"""

import time
import uuid
from dataclasses import dataclass, field
from threading import Lock


SESSION_TTL_SECONDS = 60 * 60  # 1 hour of inactivity -> session dropped


@dataclass
class Session:
    session_id: str
    site_url: str
    products: list
    policies: dict
    meta: dict
    chat_history: list = field(default_factory=list)  # [{"role": "user"/"assistant", "content": str}]
    last_active: float = field(default_factory=time.time)

    def touch(self):
        self.last_active = time.time()


class SessionStore:
    def __init__(self):
        self._sessions: dict[str, Session] = {}
        self._lock = Lock()

    def create(self, site_url: str, products: list, policies: dict, meta: dict) -> Session:
        session_id = str(uuid.uuid4())
        session = Session(
            session_id=session_id,
            site_url=site_url,
            products=products,
            policies=policies,
            meta=meta,
        )
        with self._lock:
            self._sessions[session_id] = session
        return session

    def get(self, session_id: str) -> Session | None:
        with self._lock:
            session = self._sessions.get(session_id)
        if session:
            session.touch()
        return session

    def append_message(self, session_id: str, role: str, content: str) -> None:
        session = self.get(session_id)
        if session:
            session.chat_history.append({"role": role, "content": content})

    def delete(self, session_id: str) -> None:
        with self._lock:
            self._sessions.pop(session_id, None)

    def cleanup_expired(self) -> int:
        """Remove sessions inactive past SESSION_TTL_SECONDS. Returns count removed."""
        now = time.time()
        with self._lock:
            expired = [
                sid for sid, s in self._sessions.items()
                if now - s.last_active > SESSION_TTL_SECONDS
            ]
            for sid in expired:
                del self._sessions[sid]
        return len(expired)

    def active_count(self) -> int:
        with self._lock:
            return len(self._sessions)


# Module-level singleton — import this from the FastAPI app
store = SessionStore()


In [ ]:
%%writefile groq_client.py
"""
groq_client.py
==============
Thin wrapper around the Groq API for two things this project needs:

1. Streaming chat completions — used for the conversational Q&A
   (policies, general questions about the site's catalog).
2. Vision tagging — used for image-based recommendations: user uploads
   a photo, we ask a Groq vision model to describe it in structured tags
   (jewellery type, material, style, color), then match those tags against
   the session's scraped catalog.

Requires env var GROQ_API_KEY (get one free at console.groq.com).

Colab setup:
    !pip install groq -q
    import os
    os.environ["GROQ_API_KEY"] = "your-key-here"
"""

import base64
import json
import os

from groq import AsyncGroq

TEXT_MODEL = "openai/gpt-oss-120b"       # Groq's current general-purpose text model (as of July 2026)
VISION_MODEL = "qwen/qwen3.6-27b"        # Groq's current multimodal/vision model (preview tier — verify at console.groq.com/docs/models before deploying, Groq's lineup changes often)

_client: AsyncGroq | None = None


def get_client() -> AsyncGroq:
    global _client
    if _client is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise RuntimeError("GROQ_API_KEY environment variable is not set")
        _client = AsyncGroq(api_key=api_key)
    return _client


# ---------------------------------------------------------------------------
# Chat (streaming)
# ---------------------------------------------------------------------------

def build_system_prompt(site_url: str, policies: dict, product_count: int) -> str:
    policy_context = "\n".join(
        f"{name.replace('_', ' ').title()}: {text[:1500]}"
        for name, text in policies.items() if text
    ) or "No policy information was found for this site."

    return f"""You are a helpful shopping assistant for the jewellery website {site_url}.
You ONLY answer questions about this specific website — its products, policies, and shopping experience.
The site has {product_count} products in the current catalog.

Here is the policy information scraped from this site:
{policy_context}

Rules:
- Answer policy questions (returns, shipping, etc.) using ONLY the policy information above. If something isn't covered, say you don't have that information and suggest the user check the site directly.
- Keep answers concise and friendly, like a helpful store assistant.
- If asked for product recommendations, mention that the user can use the image upload or the guided filter (occasion/price/material/type) for the best matches.
- Do not make up information about products or policies that isn't provided to you.
"""


async def stream_chat_response(session_id: str, system_prompt: str, chat_history: list, user_message: str):
    """
    Async generator yielding text chunks as they arrive from Groq.
    chat_history: list of {"role": "user"/"assistant", "content": str}
    """
    client = get_client()
    messages = [{"role": "system", "content": system_prompt}]
    messages.extend(chat_history[-10:])  # keep last 10 turns for context, bound token usage
    messages.append({"role": "user", "content": user_message})

    stream = await client.chat.completions.create(
        model=TEXT_MODEL,
        messages=messages,
        stream=True,
        temperature=0.5,
        max_tokens=800,
    )
    async for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta


# ---------------------------------------------------------------------------
# Vision tagging (image-based recommendations)
# ---------------------------------------------------------------------------

VISION_TAG_PROMPT = """Look at this jewellery image and respond with ONLY a JSON object (no markdown, no explanation) in this exact format:
{
  "jewellery_type": "ring" | "necklace" | "earrings" | "bracelet" | "bangle" | "pendant" | "anklet" | "other",
  "material": "gold" | "silver" | "platinum" | "diamond" | "pearl" | "gemstone" | "other",
  "style": "a few descriptive words, e.g. 'minimalist', 'vintage', 'statement', 'floral'",
  "color": "dominant color(s)",
  "keywords": ["list", "of", "5-8", "descriptive", "keywords", "for", "matching", "similar", "products"]
}
Respond with ONLY the JSON object."""


async def tag_image(image_bytes: bytes, mime_type: str = "image/jpeg") -> dict:
    """Send an image to Groq's vision model and return structured tags."""
    client = get_client()
    b64_image = base64.b64encode(image_bytes).decode("utf-8")

    response = await client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": VISION_TAG_PROMPT},
                    {"type": "image_url", "image_url": {"url": f"data:{mime_type};base64,{b64_image}"}},
                ],
            }
        ],
        temperature=0.2,
        max_tokens=600,
        reasoning_effort="none",  # qwen3 models only accept 'none' or 'default' — 'low'/'medium'/'high' are GPT-OSS-only and cause a 400. 'none' disables reasoning entirely, which is fine here since tagging doesn't need chain-of-thought.
    )

    raw = response.choices[0].message.content.strip()
    print(f"[groq_client] raw vision response: {raw[:500]!r}")  # temporary debug log — remove once this is confirmed stable

    tags = _extract_json_object(raw)
    if tags is not None:
        return tags

    return {
        "jewellery_type": "other",
        "material": "other",
        "style": "",
        "color": "",
        "keywords": [],
    }


def _extract_json_object(raw: str) -> dict | None:
    """
    Robustly pull a JSON object out of a model response that may contain
    markdown fences, stray reasoning text, or prose before/after the JSON.
    """
    # 1. try straight parse first (fast path when the model behaves)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # 2. strip markdown code fences
    cleaned = raw
    if "```" in cleaned:
        fence_match = re.search(r"```(?:json)?\s*(.*?)```", cleaned, re.DOTALL)
        if fence_match:
            cleaned = fence_match.group(1).strip()
            try:
                return json.loads(cleaned)
            except json.JSONDecodeError:
                pass

    # 3. find the first {...} block anywhere in the text (handles leaked reasoning text)
    brace_match = re.search(r"\{.*\}", raw, re.DOTALL)
    if brace_match:
        try:
            return json.loads(brace_match.group(0))
        except json.JSONDecodeError:
            pass

    return None


In [ ]:
%%writefile matching.py
"""
matching.py
===========
Pure in-memory filtering/matching over a session's product list. No DB,
no vector search — just Python over a list of dicts, which is plenty fast
for realistic jewellery-catalog sizes (hundreds to low thousands of items).

Two entry points:
  - filter_products(...)  -> MCQ-style guided recommendation (occasion,
    price range, material, jewellery type)
  - match_products_by_image_tags(...) -> image-based recommendation,
    scores products against the tags Groq's vision model extracted
"""

import re

OCCASION_KEYWORDS = {
    "wedding": ["wedding", "bridal", "bride", "marriage"],
    "engagement": ["engagement", "proposal", "solitaire"],
    "party": ["party", "cocktail", "statement", "evening"],
    "daily wear": ["daily", "everyday", "casual", "minimal"],
    "festive": ["festive", "festival", "traditional", "ethnic"],
    "gift": ["gift", "anniversary", "birthday"],
}


def filter_products(
    products: list,
    occasion: str | None = None,
    min_price: float | None = None,
    max_price: float | None = None,
    material: str | None = None,
    jewellery_type: str | None = None,
    limit: int = 20,
) -> list:
    """MCQ-style guided filter. Any argument left as None is not applied."""
    results = products

    if min_price is not None:
        results = [p for p in results if p.get("price") is not None and p["price"] >= min_price]

    if max_price is not None:
        results = [p for p in results if p.get("price") is not None and p["price"] <= max_price]

    if material:
        material_lower = material.lower()
        results = [
            p for p in results
            if material_lower in (p.get("title", "") + " " + p.get("description", "")).lower()
        ]

    if jewellery_type:
        type_lower = jewellery_type.lower()
        results = [
            p for p in results
            if type_lower in (p.get("title", "") + " " + p.get("description", "")).lower()
        ]

    if occasion:
        keywords = OCCASION_KEYWORDS.get(occasion.lower(), [occasion.lower()])
        results = [
            p for p in results
            if any(kw in (p.get("title", "") + " " + p.get("description", "")).lower() for kw in keywords)
        ]

    return results[:limit]


def match_products_by_image_tags(products: list, tags: dict, limit: int = 12) -> list:
    """
    Score each product against the tags extracted from an uploaded image
    (jewellery_type, material, style, color, keywords) and return the
    best matches, highest score first.
    """
    search_terms = []
    for field in ("jewellery_type", "material", "style", "color"):
        value = tags.get(field, "")
        if value and value != "other":
            search_terms.append(value.lower())
    search_terms.extend(k.lower() for k in tags.get("keywords", []))

    if not search_terms:
        return products[:limit]

    scored = []
    for p in products:
        haystack = (p.get("title", "") + " " + p.get("description", "")).lower()
        score = sum(1 for term in search_terms if term in haystack)
        # bonus weight if jewellery_type matches exactly — this is the strongest signal
        jtype = tags.get("jewellery_type", "").lower()
        if jtype and jtype != "other" and jtype in haystack:
            score += 2
        if score > 0:
            scored.append((score, p))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = [p for _, p in scored[:limit]]

    # fallback: if nothing scored, just return a slice so the UI isn't empty
    return results if results else products[:limit]


def get_available_filter_options(products: list) -> dict:
    """
    Inspect the catalog to suggest sensible MCQ options for the frontend
    (e.g. actual price range present, rather than hardcoded guesses).
    """
    prices = [p["price"] for p in products if p.get("price") is not None]
    return {
        "min_price": min(prices) if prices else 0,
        "max_price": max(prices) if prices else 0,
        "occasions": list(OCCASION_KEYWORDS.keys()),
        "jewellery_types": ["ring", "necklace", "earrings", "bracelet", "bangle", "pendant", "anklet"],
        "materials": ["gold", "silver", "platinum", "diamond", "pearl", "gemstone"],
    }


In [ ]:
%%writefile main.py
"""
main.py
=======
FastAPI backend for the jewellery chatbot. Serves:
  - POST /api/session          -> submit a site URL, scrape it, get a session_id
  - GET  /api/session/{id}     -> session status + available filter options
  - POST /api/chat             -> SSE streaming chat response
  - POST /api/image-search     -> upload an image, get matched products
  - POST /api/filter           -> MCQ-style guided filter
  - (in production) serves the built React frontend as static files

Run locally:
    uvicorn main:app --reload --port 8000

Colab (for quick endpoint testing via ngrok, NOT for real deployment):
    !pip install fastapi uvicorn python-multipart nest-asyncio pyngrok -q
    import nest_asyncio; nest_asyncio.apply()
    # then run uvicorn.run(app, port=8000) in a cell and expose via pyngrok
"""

import asyncio
import os
from contextlib import asynccontextmanager

from fastapi import FastAPI, HTTPException, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import StreamingResponse
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel

from scraper import scrape_site
from session_store import store
from groq_client import build_system_prompt, stream_chat_response, tag_image
from matching import filter_products, match_products_by_image_tags, get_available_filter_options


# ---------------------------------------------------------------------------
# Background session cleanup
# ---------------------------------------------------------------------------

async def _cleanup_loop():
    while True:
        await asyncio.sleep(600)  # every 10 minutes
        store.cleanup_expired()


@asynccontextmanager
async def lifespan(app: FastAPI):
    task = asyncio.create_task(_cleanup_loop())
    yield
    task.cancel()


app = FastAPI(title="Jewellery Site Chatbot API", lifespan=lifespan)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # tighten this to your actual frontend origin in production
    allow_methods=["*"],
    allow_headers=["*"],
)


# ---------------------------------------------------------------------------
# Request/response models
# ---------------------------------------------------------------------------

class CreateSessionRequest(BaseModel):
    url: str


class ChatRequest(BaseModel):
    session_id: str
    message: str


class FilterRequest(BaseModel):
    session_id: str
    occasion: str | None = None
    min_price: float | None = None
    max_price: float | None = None
    material: str | None = None
    jewellery_type: str | None = None


# ---------------------------------------------------------------------------
# Session lifecycle
# ---------------------------------------------------------------------------

@app.post("/api/session")
async def create_session(req: CreateSessionRequest):
    if not req.url or not req.url.strip():
        raise HTTPException(status_code=400, detail="URL is required")

    result = await scrape_site(req.url)

    if not result["products"]:
        raise HTTPException(
            status_code=422,
            detail="Could not find any products on this site. It may block automated access, "
                   "or use a catalog structure this scraper doesn't recognize yet.",
        )

    session = store.create(
        site_url=result["site_url"],
        products=result["products"],
        policies=result["policies"],
        meta=result["meta"],
    )

    return {
        "session_id": session.session_id,
        "site_url": session.site_url,
        "product_count": len(session.products),
        "scrape_method": session.meta["method"],
        "elapsed_seconds": session.meta["elapsed_seconds"],
        "filter_options": get_available_filter_options(session.products),
    }


@app.get("/api/session/{session_id}")
async def get_session(session_id: str):
    session = store.get(session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found or expired")
    return {
        "session_id": session.session_id,
        "site_url": session.site_url,
        "product_count": len(session.products),
        "filter_options": get_available_filter_options(session.products),
    }


# ---------------------------------------------------------------------------
# Chat (SSE streaming)
# ---------------------------------------------------------------------------

@app.post("/api/chat")
async def chat(req: ChatRequest):
    session = store.get(req.session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found or expired")

    system_prompt = build_system_prompt(session.site_url, session.policies, len(session.products))
    store.append_message(req.session_id, "user", req.message)

    async def event_stream():
        full_response = ""
        try:
            async for chunk in stream_chat_response(
                req.session_id, system_prompt, session.chat_history, req.message
            ):
                full_response += chunk
                yield f"data: {chunk}\n\n"
        finally:
            store.append_message(req.session_id, "assistant", full_response)
            yield "event: done\ndata: \n\n"

    return StreamingResponse(event_stream(), media_type="text/event-stream")


# ---------------------------------------------------------------------------
# Image-based recommendation
# ---------------------------------------------------------------------------

@app.post("/api/image-search")
async def image_search(session_id: str = Form(...), image: UploadFile = File(...)):
    session = store.get(session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found or expired")

    image_bytes = await image.read()
    if len(image_bytes) > 10 * 1024 * 1024:
        raise HTTPException(status_code=413, detail="Image too large (max 10MB)")

    tags = await tag_image(image_bytes, mime_type=image.content_type or "image/jpeg")
    matches = match_products_by_image_tags(session.products, tags)

    return {"tags": tags, "matches": matches}


# ---------------------------------------------------------------------------
# MCQ-style guided filter
# ---------------------------------------------------------------------------

@app.post("/api/filter")
async def filter_endpoint(req: FilterRequest):
    session = store.get(req.session_id)
    if not session:
        raise HTTPException(status_code=404, detail="Session not found or expired")

    matches = filter_products(
        session.products,
        occasion=req.occasion,
        min_price=req.min_price,
        max_price=req.max_price,
        material=req.material,
        jewellery_type=req.jewellery_type,
    )
    return {"matches": matches}


# ---------------------------------------------------------------------------
# Serve built React frontend (static files) — added once frontend is built
# ---------------------------------------------------------------------------

FRONTEND_DIST = os.path.join(os.path.dirname(__file__), "static")
if os.path.isdir(FRONTEND_DIST):
    app.mount("/", StaticFiles(directory=FRONTEND_DIST, html=True), name="frontend")


@app.get("/api/health")
async def health():
    return {"status": "ok", "active_sessions": store.active_count()}


## 3. Set your Groq API key
Get a free key at [console.groq.com](https://console.groq.com)

In [ ]:
import os
os.environ["GROQ_API_KEY"] = "PASTE_YOUR_GROQ_API_KEY_HERE"

## 4. Start the server
Runs FastAPI in a background thread inside this Colab session.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn, threading, time, asyncio
from main import app

def run_server():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    config = uvicorn.Config(app, host="0.0.0.0", port=8000, log_level="info")
    server = uvicorn.Server(config)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

time.sleep(5)

import requests
r = requests.get("http://localhost:8000/api/health")
print(r.status_code, r.json())

## 5. Test — create a session (scrapes a real site)
Swap the URL for whichever jewellery site you want to test against.

In [ ]:
import requests

r = requests.post("http://localhost:8000/api/session", json={"url": "https://www.giva.co"})
print(r.status_code)
print(r.json())
session_id = r.json()["session_id"]

## 6. Test — chat (streams a policy answer via Groq)

In [ ]:
r = requests.post(
    "http://localhost:8000/api/chat",
    json={"session_id": session_id, "message": "What is your return policy?"},
    stream=True,
)
for line in r.iter_lines():
    if line:
        print(line.decode())

## 7. Test — MCQ-style guided filter (no DB, pure in-memory)

In [ ]:
import json

r = requests.post(
    "http://localhost:8000/api/filter",
    json={"session_id": session_id, "jewellery_type": "ring", "max_price": 5000},
)
print(json.dumps(r.json(), indent=2)[:1500])

## 8. Test — image-based recommendation
Upload a jewellery photo first (Colab file browser on the left, or use `files.upload()` below),
then run the next cell with the matching filename.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick a jewellery photo from your computer
image_filename = list(uploaded.keys())[0]
print("Uploaded:", image_filename)

In [ ]:
with open(image_filename, "rb") as f:
    r = requests.post(
        "http://localhost:8000/api/image-search",
        data={"session_id": session_id},
        files={"image": f},
    )
print(json.dumps(r.json(), indent=2)[:2000])

## Next steps
Once all of the above return sensible results (real product counts, streamed
chat text, filtered matches, and real image tags), the backend is fully
validated. Copy `scraper.py`, `session_store.py`, `groq_client.py`,
`matching.py`, `main.py`, and `requirements.txt` into your VS Code project
folder for the frontend build and deployment.